In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

master = pd.read_csv(
    "../data/processed/tourism_master_cleaned.csv"
)

print("Dataset shape:", master.shape)
display(master.head())

Dataset shape: (52930, 27)


,transactionid,userid,visityear,visitmonth,visitmode,attractionid,rating,user_continentid,user_regionid,user_countryid,user_cityid,user_cityname,user_city_countryid,user_country,user_country_regionid,user_region,user_region_continentid,user_continent,attractioncityid,attractiontypeid,attraction,attractionaddress,attractiontype,attraction_cityname,attraction_countryid,attraction_country,attraction_regionid
0,3,70456,2022,10,2,640,5,5,21,163,4341.0,Guildford,109.0,United Kingdom,21,Western Europe,5,Europe,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala,1,Cameroon,1
1,8,7567,2022,10,4,640,5,2,8,48,464.0,Ontario,48.0,Canada,8,Northern America,2,America,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala,1,Cameroon,1
2,9,79069,2022,10,3,640,5,2,9,54,774.0,Brazil,51.0,Brazil,9,South America,2,America,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala,1,Cameroon,1
3,10,31019,2022,10,3,640,3,5,17,135,583.0,Zurich,48.0,Switzerland,17,Central Europe,5,Europe,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala,1,Cameroon,1
4,15,43611,2022,10,2,640,3,5,21,163,1396.0,Manchester,51.0,United Kingdom,21,Western Europe,5,Europe,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,Douala,1,Cameroon,1


In [3]:
master["visit_date"] = pd.to_datetime(
    master["visityear"].astype(str) + "-" +
    master["visitmonth"].astype(str) + "-01"
)

master["visit_quarter"] = master["visit_date"].dt.quarter

master["visit_month_name"] = (
    master["visit_date"]
    .dt.month_name()
)

print(
    master[
        [
            "visityear",
            "visitmonth",
            "visit_date",
            "visit_quarter",
            "visit_month_name"
        ]
    ].head()
)

   visityear  visitmonth visit_date  visit_quarter visit_month_name
0       2022          10 2022-10-01              4          October
1       2022          10 2022-10-01              4          October
2       2022          10 2022-10-01              4          October
3       2022          10 2022-10-01              4          October
4       2022          10 2022-10-01              4          October


In [4]:
attraction_stats = (
    master.groupby("attractionid")
    .agg(
        attraction_visit_count=("transactionid", "count"),
        attraction_avg_rating=("rating", "mean"),
        attraction_rating_std=("rating", "std")
    )
    .reset_index()
)

attraction_stats["attraction_rating_std"] = (
    attraction_stats["attraction_rating_std"]
    .fillna(0)
)

display(attraction_stats.head())

,attractionid,attraction_visit_count,attraction_avg_rating,attraction_rating_std
0,369,2765,3.415190,1.288006
1,481,2104,4.275665,0.866263
2,640,13198,4.267086,0.873768
3,650,3044,3.976347,0.981642
4,673,2914,3.800618,1.125670


In [5]:
master = master.merge(
    attraction_stats,
    on="attractionid",
    how="left",
    validate="many_to_one"
)

print(master.shape)

(52930, 33)


In [6]:
user_stats = (
    master.groupby("userid")
    .agg(
        user_transaction_count=("transactionid", "count"),
        user_unique_attractions=("attractionid", "nunique"),
        user_avg_rating=("rating", "mean")
    )
    .reset_index()
)

display(user_stats.head())

,userid,user_transaction_count,user_unique_attractions,user_avg_rating
0,14,3,2,4.666667
1,16,10,5,4.700000
2,20,1,1,4.000000
3,23,1,1,5.000000
4,25,1,1,5.000000


In [7]:
master = master.merge(
    user_stats,
    on="userid",
    how="left",
    validate="many_to_one"
)

print("Shape:", master.shape)

Shape: (52930, 36)


In [8]:
type_stats = (
    master.groupby("attractiontypeid")
    .agg(
        type_visit_count=("transactionid", "count"),
        type_avg_rating=("rating", "mean")
    )
    .reset_index()
)

display(type_stats)

,attractiontypeid,type_visit_count,type_avg_rating
0,2,581,4.058520
1,10,377,4.180371
2,13,10917,3.845745
3,19,135,4.503704
4,34,1324,3.869335
5,44,798,3.538847
6,45,978,4.256646
7,61,511,4.416830
8,63,13251,4.266848
9,64,44,4.318182


In [9]:
master = master.merge(
    type_stats,
    on="attractiontypeid",
    how="left",
    validate="many_to_one"
)

In [10]:
print(master.columns.tolist())

['transactionid', 'userid', 'visityear', 'visitmonth', 'visitmode', 'attractionid', 'rating', 'user_continentid', 'user_regionid', 'user_countryid', 'user_cityid', 'user_cityname', 'user_city_countryid', 'user_country', 'user_country_regionid', 'user_region', 'user_region_continentid', 'user_continent', 'attractioncityid', 'attractiontypeid', 'attraction', 'attractionaddress', 'attractiontype', 'attraction_cityname', 'attraction_countryid', 'attraction_country', 'attraction_regionid', 'visit_date', 'visit_quarter', 'visit_month_name', 'attraction_visit_count', 'attraction_avg_rating', 'attraction_rating_std', 'user_transaction_count', 'user_unique_attractions', 'user_avg_rating', 'type_visit_count', 'type_avg_rating']


In [11]:
id_columns = [
    "transactionid",
    "userid",
    "attractionid",
    "user_cityid",
    "user_countryid",
    "user_regionid",
    "user_continentid",
    "attractioncityid",
    "attractiontypeid",
    "attraction_countryid",
    "attraction_regionid"
]

In [12]:
id_columns = [
    col for col in id_columns
    if col in master.columns
]

print("ID columns:")
print(id_columns)

ID columns:
['transactionid', 'userid', 'attractionid', 'user_cityid', 'user_countryid', 'user_regionid', 'user_continentid', 'attractioncityid', 'attractiontypeid', 'attraction_countryid', 'attraction_regionid']


In [13]:
regression_target = "rating"

In [14]:
classification_target = "visitmode_name"

In [15]:
recommendation_data = master[
    [
        "userid",
        "attractionid",
        "attraction",
        "attractiontype",
        "rating"
    ]
].copy()

display(recommendation_data.head())

,userid,attractionid,attraction,attractiontype,rating
0,70456,640,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5
1,7567,640,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5
2,79069,640,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5
3,31019,640,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,3
4,43611,640,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,3


In [20]:
display(recommendation_data.shape)

(52930, 5)

In [16]:
print(
    "Unique users:",
    recommendation_data["userid"].nunique()
)

print(
    "Unique attractions:",
    recommendation_data["attractionid"].nunique()
)

Unique users: 33530
Unique attractions: 30


In [17]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [21]:
master.to_csv(
    "../data/processed/tourism_features.csv",
    index=False
)

recommendation_data.to_csv(
    "../data/processed/recommendation_data.csv",
    index=False
)

print("✅ Feature-engineered datasets saved.")

✅ Feature-engineered datasets saved.
